# MicroSplit demo: HT-LIF24, 5 ms, 2 channels

This notebook splits one superimposed microscopy image into two structures:

- Channel 0: nucleus.
- Channel 1: microtubules.

The network input is the real superimposed channel of a 5 ms exposure. The 5 ms data has much noise. The evaluation uses the 500 ms capture of the same frames as the ground truth.

The notebook has three parts:

1. Train the noise models: Noise2Void (N2V) gives a signal estimate, then a Gaussian mixture model (GMM) fits the noise for each target channel.
2. Train MicroSplit with the noise models in the loss (denoiSplit).
3. Predict on the validation split and compare the result with the 500 ms ground truth.

The notebook follows `scripts/microsplit_lif24_5ms.py`. It uses one GPU and no SLURM.

## 1. Imports

Import the libraries. The notebook uses the CAREamics Lightning API for MicroSplit and the `CAREamist` API for N2V.

In [ ]:
from pathlib import Path

import lightning.pytorch as L
import matplotlib.pyplot as plt
import numpy as np
import tifffile
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar
from torch.utils.data import DataLoader
from torch.utils.data._utils.collate import default_collate

from careamics import CAREamist
from careamics.config import (
    GaussianMixtureNMConfig,
    MicroSplitDataConfig,
    MultiChannelNMConfig,
    create_advanced_microsplit_config,
    create_n2v_config,
)
from careamics.dataset.factory import (
    PairedInputTarget,
    create_microsplit_dataset,
    create_microsplit_pred_dataset,
)
from careamics.dataset.factory.factory import TrainValData
from careamics.lightning.modules.microsplit_module import MicroSplitModule
from careamics.lightning.utils.load_checkpoint import load_module_from_checkpoint
from careamics.lightning.prediction.convert_prediction import convert_prediction
from careamics.lvae_training.dataset.utils.data_utils import get_datasplit_tuples
from careamics.lvae_training.metrics import compute_stats
from careamics.noise_model import NoiseModelTrainer

## 2. Parameters

Set the paths and the training budget here. All other cells read these values.

- `DATA_ROOT` contains the `5ms/` and `500ms/` folders. Each folder contains `Set1` to `Set6`.
- `CH_IDX_LIST` selects the raw channels: the targets first, then the superimposed input.
- `NM_PATHS`: if you already have fitted noise models, set the two `.npz` paths. The notebook then skips N2V and the GMM fit.

The epoch counts are small for a demo. The script uses 40 MicroSplit epochs for the reported results. Increase `NUM_EPOCHS` to get comparable metrics.

`USE_PRETRAINED` selects one of two modes:

- `True`: the notebook skips N2V, the GMM fit and the MicroSplit training. It loads the pretrained checkpoint 
- `False`: the notebook trains all the models.

In [ ]:
DATA_ROOT = Path("/group/jug/public_html/microsplit/ht_lif24_tiff")
EXPOSURE = "5ms"            # noisy input and training data
HIGHSNR_EXPOSURE = "500ms"  # ground truth for the evaluation
CH_IDX_LIST = [0, 1, 8]     # targets [Nucleus, MicroTubules] + superimposed input "01"
N_TARGETS = len(CH_IDX_LIST) - 1

WORK_DIR = Path("microsplit_lif24_5ms_demo").resolve()
EXPERIMENT_NAME = "ht_lif24_5ms_demo"
SEED = 42

# Data split params
VAL_FRACTION = 0.1
TEST_FRACTION = 0.1

# Noise model
NM_PATHS = None
N2V_NUM_EPOCHS = 10
NM_N_GAUSSIAN = 6
NM_N_COEFF = 4
NM_MIN_SIGMA = 0.125  # clamps the VARIANCE.
NM_N_EPOCHS = 2000

# MicroSplit
PATCH_SIZE = (64, 64)
GRID_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 4
NUM_EPOCHS = 10
Z_DIMS = [128] * 4
N_FILTERS = 64
MULTISCALE_COUNT = 3
NOISE_MODEL_LIKELIHOOD_WEIGHT = 0.9
GAUSSIAN_LIKELIHOOD_WEIGHT = 0.1

# Prediction
MMSE_COUNT = 50   # number of posterior samples to average; the paper uses 50
EVAL_SPLIT = "test"  # "test" = the 10 paper test frames, "val" = the 12 validation frames

# Pretrained model (our paper reproduction, 400 epochs)
USE_PRETRAINED = True
PRETRAINED_DIR = Path("/home/igor.zubarev/projects/careamics/scripts")
PRETRAINED_CKPT = (
    PRETRAINED_DIR / "lvae_checkpoints/5ms/ht_lif24_5ms_ngds_v3_400ep/checkpoints/last.ckpt"
)

PRETRAINED_NM_PATHS = [
    PRETRAINED_DIR / f"noise_models/5ms_ngds_ht_lif24_5ms_ngds_v3_400ep/noise_model_ch{c}.npz"
    for c in range(2)
]
if USE_PRETRAINED:
    NM_PATHS = PRETRAINED_NM_PATHS

WORK_DIR.mkdir(parents=True, exist_ok=True)
L.seed_everything(SEED, workers=True)

## 3. Load the data

Each TIFF file has the shape `(frames, raw channels, Y, X)`. The function below keeps the three channels from `CH_IDX_LIST` and joins the six Sets. The result has the shape `(112, 3, Y, X)`.

`get_datasplit_tuples` gives the same fixed split as the original MicroSplit code:

- Validation: 12 frames.
- Test: 10 frames.
- Training: the remaining 90 frames.

The 500 ms stack uses the same frame indices. Thus frame `k` of the 5 ms data and frame `k` of the 500 ms data show the same field of view.

In [ ]:
def load_exposure(exposure: str) -> np.ndarray:
    # Return all frames of Set1..Set6 as float32 SCYX, channels = CH_IDX_LIST.
    stacks = [
        tifffile.imread(DATA_ROOT / exposure / f"Set{i}" / f"uSplit_{exposure}.tif")[
            :, CH_IDX_LIST
        ]
        for i in range(1, 7)
    ]
    return np.concatenate(stacks, axis=0).astype(np.float32)


data = load_exposure(EXPOSURE)
train_idx, val_idx, test_idx = get_datasplit_tuples(
    VAL_FRACTION, TEST_FRACTION, len(data)
)
print(f"{EXPOSURE} data: {data.shape}")
print(f"train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)} frames")

# The network input is the last channel; the targets are the other channels.
train_input, train_target = data[train_idx, -1:], data[train_idx, :-1]
val_input, val_target = data[val_idx, -1:], data[val_idx, :-1]

Show one training frame. The first panel is the network input. The other panels are the two target channels at 5 ms.

In [ ]:
frame = 0
titles = ["Input (superimposed)", "Target 0: Nucleus", "Target 1: MicroTubules"]
images = [train_input[frame, 0], train_target[frame, 0], train_target[frame, 1]]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap="magma")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

## 4. Noise model training

The denoiSplit loss needs a noise model for each target channel. A noise model gives the probability of a noisy pixel value for a given clean signal value.

We do not have clean 5 ms data. Thus we use two steps:

1. Train N2V on the noisy training data. The N2V prediction is the signal estimate.
2. Fit one GMM for each target channel on the pairs (signal estimate, noisy observation).

Use only the training split for these steps. If `NM_PATHS` is set (for example, with `USE_PRETRAINED = True`), the cells in 4.1 and 4.2 do nothing.

### 4.1 Train N2V

N2V is self-supervised. It uses only the noisy 5 ms data. The N2V model trains on all three channels of the training split. The prediction has the same `SYXC` layout as the input.

In [ ]:
nm_dir = WORK_DIR / "noise_models"

if NM_PATHS is None:
    nm_observation = np.moveaxis(data[train_idx], 1, -1)  # SCYX -> SYXC

    n2v_config = create_n2v_config(
        experiment_name=f"{EXPERIMENT_NAME}_n2v",
        data_type="array",
        axes="SYXC",
        n_channels=len(CH_IDX_LIST),
        patch_size=PATCH_SIZE,
        batch_size=BATCH_SIZE,
        num_epochs=N2V_NUM_EPOCHS,
    )
    n2v_config.data_config.seed = SEED
    n2v = CAREamist(config=n2v_config, work_dir=str(nm_dir))
    n2v.train(train_data=nm_observation)

    n2v_prediction, _ = n2v.predict(nm_observation, tile_size=(256, 256))
    nm_signal = np.concatenate(n2v_prediction, axis=0)  # SYXC
    print(f"N2V signal estimate: {nm_signal.shape}")

Compare the noisy data with the N2V signal estimate for one target channel. The signal estimate must show the same structures with less noise.

In [ ]:
if NM_PATHS is None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(nm_observation[frame, ..., 1], cmap="magma")
    axes[0].set_title("Noisy 5 ms, MicroTubules")
    axes[1].imshow(nm_signal[frame, ..., 1], cmap="magma")
    axes[1].set_title("N2V signal estimate")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()

### 4.2 Fit the GMM noise models

Fit one GMM for each target channel. The input channel does not get a noise model, because the loss compares the prediction only with the targets.

The values of `NM_N_GAUSSIAN`, `NM_N_COEFF` and `NM_MIN_SIGMA` are the values of the original MicroSplit noise models.

Two cautions:

- `min_sigma` clamps the variance. A large value (for example 200) makes the GMM a constant Gaussian on this data.
- An N2V model with few epochs can predict values below 0. The GMM configuration does not accept a negative signal. Thus clip the signal estimate at 0. Do not clip the observation.

In [ ]:
if NM_PATHS is None:
    nm_trainer = NoiseModelTrainer(
        n_gaussian=NM_N_GAUSSIAN,
        n_coeff=NM_N_COEFF,
        min_sigma=NM_MIN_SIGMA,
    )
    nm_trainer.train_from_pairs(
        signal=np.clip(nm_signal[..., :N_TARGETS], 0, None),
        observation=nm_observation[..., :N_TARGETS],
        axes="SYXC",
        n_epochs=NM_N_EPOCHS,
    )
    NM_PATHS = nm_trainer.save(nm_dir)
    print(f"Saved noise models: {NM_PATHS}")

Show the fit loss of each GMM. The loss must decrease and then become flat. If the loss is `nan`, do not use the noise model. The trainer saves the `.npz` file also when the fit fails.

In [ ]:
if "nm_trainer" in globals():
    fig, axes = plt.subplots(1, N_TARGETS, figsize=(6 * N_TARGETS, 4))
    for ch, (ax, losses) in enumerate(zip(axes, nm_trainer.train_losses)):
        ax.plot(losses)
        ax.set_title(f"GMM fit, target channel {ch}")
        ax.set_xlabel("step")
        ax.set_ylabel("negative log-likelihood")
    plt.tight_layout()

### 4.3 Make the noise model configuration

The noise model is a field of the MicroSplit configuration. The line below reads the `.npz` files into a `MultiChannelNMConfig`. At the start of training, `MicroSplitModule` builds the noise model from this configuration and normalizes it with the training data statistics.

In [ ]:
noise_model = MultiChannelNMConfig(
    noise_models=[GaussianMixtureNMConfig.from_npz(p) for p in NM_PATHS]
)
print(f"{len(noise_model.noise_models)} noise models loaded")

## 5. MicroSplit training

### 5.1 Make the configuration

`create_advanced_microsplit_config` makes the full configuration: data, model, loss and trainer.

- The loss has two terms: the noise model likelihood (weight 0.9) and the Gaussian likelihood (weight 0.1).
- The model is a ladder VAE with 4 latent levels of 128 dimensions and 3 lateral contexts (`multiscale_count`).
- Mixed precision (16 bit) and gradient clipping at 0.5 are the values of the script.

In [ ]:
config = create_advanced_microsplit_config(
    experiment_name=EXPERIMENT_NAME,
    data_type="array",
    axes="SCYX",
    patch_size=PATCH_SIZE,
    output_channels=N_TARGETS,
    multiscale_count=MULTISCALE_COUNT,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    gaussian_likelihood_weight=GAUSSIAN_LIKELIHOOD_WEIGHT,
    noise_model_likelihood_weight=NOISE_MODEL_LIKELIHOOD_WEIGHT,
    noise_model=noise_model,
    model_params={"z_dims": Z_DIMS, "n_filters": N_FILTERS},
    num_workers=NUM_WORKERS,
    trainer_params={
        "max_epochs": NUM_EPOCHS,
        "precision": 16,
        "gradient_clip_algorithm": "value",
        "gradient_clip_val": 0.5,
    },
    seed=SEED,
)
loss = config.algorithm_config.loss
print(f"loss weights: noise model {loss.noise_model_likelihood_weight}, "
      f"gaussian {loss.gaussian_likelihood_weight}")

### 5.2 Make the data module

The data module gives the patches to the Lightning `Trainer`.

- The training dataset calculates the normalization statistics.
- The validation dataset uses the same statistics as the training dataset.
- The prediction dataset cuts each frame into overlapping tiles. The prediction step uses these tiles.

In [ ]:
class MicroSplitDataModule(L.LightningDataModule):
    def __init__(
        self,
        train_config: MicroSplitDataConfig,
        train_input: np.ndarray,
        train_target: np.ndarray,
        val_input: np.ndarray,
        val_target: np.ndarray,
        batch_size: int,
        num_workers: int,
    ) -> None:
        super().__init__()
        self.train_config = train_config
        self.val_config = train_config.convert_mode("validating")
        self.pred_config: MicroSplitDataConfig | None = None
        self.train_input = [train_input]
        self.train_target = [train_target]
        self.val_input = [val_input]
        self.val_target = [val_target]
        self.pred_input: list[np.ndarray] | None = None
        self.batch_size = batch_size
        self.num_workers = num_workers
        self._data = TrainValData(
            train_data=self.train_input,
            val_data=self.val_input,
            train_data_target=self.train_target,
            val_data_target=self.val_target,
        )
        self.train_dataset = None
        self.val_dataset = None
        self.predict_dataset = None

    def setup(self, stage: str) -> None:
        if stage in ("fit", "validate"):
            if self.train_dataset is None:
                self.train_dataset = create_microsplit_dataset(
                    config=self.train_config,
                    data=PairedInputTarget(
                        input_data=self.train_input, target_data=self.train_target
                    ),
                )
            if self.val_dataset is None:
                val_cfg = self.val_config.model_copy(
                    update={"normalization": self.train_config.normalization}
                )
                self.val_dataset = create_microsplit_dataset(
                    config=val_cfg,
                    data=PairedInputTarget(
                        input_data=self.val_input, target_data=self.val_target
                    ),
                )
        elif stage == "predict":
            self.predict_dataset = create_microsplit_pred_dataset(
                config=self.pred_config, input_data=self.pred_input
            )

    def _loader(self, dataset, shuffle: bool) -> DataLoader:
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=shuffle,
            collate_fn=default_collate,
        )

    def train_dataloader(self):
        return self._loader(self.train_dataset, shuffle=True)

    def val_dataloader(self):
        return self._loader(self.val_dataset, shuffle=False)

    def predict_dataloader(self):
        return self._loader(self.predict_dataset, shuffle=False)


dm = MicroSplitDataModule(
    train_config=config.data_config,
    train_input=train_input,
    train_target=train_target,
    val_input=val_input,
    val_target=val_target,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

### 5.3 Make the model and the trainer

`MicroSplitModule` contains the ladder VAE, the loss and the noise model.

The trainer keeps two checkpoints in `WORK_DIR/checkpoints`:

- `last.ckpt`: the weights after the last epoch.
- `best.ckpt`: the weights with the lowest validation loss.

In [ ]:
model = MicroSplitModule(config.algorithm_config)

trainer_params = config.training_config.trainer_params
ckpt_dir = WORK_DIR / "checkpoints"
trainer = Trainer(
    max_epochs=trainer_params["max_epochs"],
    precision=trainer_params["precision"],
    gradient_clip_algorithm=trainer_params["gradient_clip_algorithm"],
    gradient_clip_val=trainer_params["gradient_clip_val"],
    default_root_dir=WORK_DIR,
    callbacks=[
        ModelCheckpoint(dirpath=ckpt_dir, filename=EXPERIMENT_NAME, save_last=True),
        ModelCheckpoint(
            dirpath=ckpt_dir, filename="best", monitor="val_loss", mode="min"
        ),
        TQDMProgressBar(refresh_rate=50),
    ],
    logger=False,
)

### 5.4 Train, or load the pretrained model

If `USE_PRETRAINED` is `False`, this cell starts the training. On one GPU, one epoch takes some time.

If `USE_PRETRAINED` is `True`, this cell loads the model from the checkpoint. `load_module_from_checkpoint` reads the algorithm configuration from the checkpoint, makes the correct Lightning module and loads the weights. The cell also makes the training dataset, because the training dataset calculates the normalization statistics. The prediction uses these statistics.

In [ ]:
if USE_PRETRAINED:
    model = load_module_from_checkpoint(PRETRAINED_CKPT)
    dm.setup("fit")
else:
    trainer.fit(model, datamodule=dm)

## 6. Prediction

### 6.1 Predict the evaluation frames

`EVAL_SPLIT` selects the frames. Use `"test"` to compare with the paper.

MicroSplit is a variational model. Each forward pass gives a different sample. The MMSE prediction is the mean of `MMSE_COUNT` samples. More samples give a better prediction, but the prediction takes more time.

The prediction dataset cuts each frame into `64 x 64` tiles. The tiles overlap, so that only the center `GRID_SIZE x GRID_SIZE` region of each tile goes into the output. `convert_prediction` joins the tiles into full frames.

In [ ]:
overlap = tuple(p - GRID_SIZE for p in PATCH_SIZE)
dm.pred_config = config.data_config.convert_mode(
    "predicting", new_patch_size=PATCH_SIZE, overlap_size=overlap
)
eval_idx = {"val": val_idx, "test": test_idx}[EVAL_SPLIT]
eval_input = data[eval_idx, -1:]
dm.pred_input = [eval_input]
dm.setup("predict")

model.n_samples = MMSE_COUNT
batches = trainer.predict(model, datamodule=dm)

# predict_step gives (prediction, uncertainty) per batch; keep the prediction.
stitched, _ = convert_prediction([b[0] for b in batches], tiled=True, restore_shape=True)
prediction = np.concatenate(stitched, axis=0) if len(stitched) > 1 else stitched[0]
if prediction.shape[1] == N_TARGETS:
    prediction = np.moveaxis(prediction, 1, -1)  # SCYX -> SYXC
print(f"prediction: {prediction.shape}")

### 6.2 Evaluate against the 500 ms ground truth

Load the target channels of the 500 ms data for the same frames. `compute_stats` gives the metrics for each channel: PSNR, range-invariant PSNR, SSIM, MS-SSIM and MicroSSIM.

In [ ]:
highsnr = load_exposure(HIGHSNR_EXPOSURE)
gt = np.moveaxis(highsnr[eval_idx, :-1], 1, -1)  # (N, Y, X, 2)

metrics = compute_stats([gt], [prediction])

### 6.3 Show the results

Show the input, the prediction and the 500 ms ground truth for one frame of the evaluation split. The top row shows the nucleus. The bottom row shows the microtubules.

The first figure shows the full frame. The second figure shows a `CROP x CROP` region, so that you can see the small structures. The crop is the region of the ground truth with the highest mean intensity.

In [ ]:
def show(frame: int, region=(slice(None), slice(None))) -> None:
    names = ["Nucleus", "MicroTubules"]
    fig, axes = plt.subplots(N_TARGETS, 3, figsize=(15, 5 * N_TARGETS))
    for ch in range(N_TARGETS):
        panels = [
            (eval_input[frame, 0], "Input (5 ms, superimposed)"),
            (prediction[frame, ..., ch], f"MicroSplit: {names[ch]}"),
            (gt[frame, ..., ch], f"500 ms GT: {names[ch]}"),
        ]
        for ax, (img, title) in zip(axes[ch], panels):
            ax.imshow(img[region], cmap="magma")
            ax.set_title(title)
            ax.axis("off")
    plt.tight_layout()
    plt.show()


def brightest_window(img: np.ndarray, size: int, step: int = 64) -> tuple[int, int]:
    # Top-left corner of the size x size window with the highest mean.
    best, pos = -np.inf, (0, 0)
    for y in range(0, img.shape[0] - size + 1, step):
        for x in range(0, img.shape[1] - size + 1, step):
            mean = img[y : y + size, x : x + size].mean()
            if mean > best:
                best, pos = mean, (y, x)
    return pos


frame = 0
CROP = 512
y0, x0 = brightest_window(gt[frame].sum(axis=-1), CROP)
show(frame)
show(frame, (slice(y0, y0 + CROP), slice(x0, x0 + CROP)))

### 6.4 Save the predictions

Save the predictions as a TIFF file with the axes `SYXC`.

In [ ]:
out_path = WORK_DIR / f"prediction_{EVAL_SPLIT}_mmse{MMSE_COUNT}.tif"
tifffile.imwrite(out_path, prediction.astype(np.float32))
print(f"Saved {out_path}")